In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import *

In [0]:
# inferschema false & header true
# Spark will automatically try to know the schema of the data True . If false dont try to read the schema of data .
flight_df = spark.read.format("csv")\
          .option("header","true")\
          .option("inferschema","false")\
          .option("mode","FAILFAST")\
          .load("/Volumes/workspace/default/tables/flight_data.csv")
flight_df.show(10)

In [0]:
flight_df.printSchema()

In [0]:
# Inferschema true

flight_df2= spark.read.format("csv")\
          .option("header","true")\
          .option("inferschema","true")\
          .option("mode","FAILFAST")\
          .load("/Volumes/workspace/default/tables/flight_data.csv")
flight_df.show(10)

In [0]:
flight_df2.printSchema()

## Schema 

In [0]:
from pyspark.sql.types import StructField, StringType, IntegerType, StructType

my_schema = StructType([
            StructField("DEST_COUNTRY_NAME",StringType(),True),
            StructField("ORIGIN_COUNTRY_NAME",StringType(),True),
            StructField("count",IntegerType(),True)
])

In [0]:
flight_df3 = spark.read.format("csv")\
    .option("header", "false")\
    .option("inferschema", "false")\
    .schema(my_schema)\
    .option("mode", "PERMISSIVE")\
    .load("/Volumes/workspace/default/tables/flight_data.csv")

flight_df3.show(5)


In [0]:
flight_df3 = spark.read.format("csv")\
    .option("header", "false")\
    .option("inferschema", "false")\
    .schema(my_schema)\
    .option("mode", "PERMISSIVE")\
    .option("SkipRows", "1")\
    .load("/Volumes/workspace/default/tables/flight_data.csv")    

flight_df3.show(5)


## Handling corrupted records in spark

In [0]:

employee_df = spark.read.format("csv")\
    .option("header", "true")\
    .option("inferschema", "true")\
    .option("mode", "PERMISSIVE")\
    .load("/Volumes/workspace/default/tables/employee_file.csv")    

employee_df.show()

In [0]:
display(employee_df)

In [0]:
employee_df.printSchema()

In [0]:
# OR
df = spark.read.format("csv").option("header", "true").load("/Volumes/workspace/default/tables/employee_file.csv")
display(df)

In [0]:
#Displaying corrrupted record in corrupted_record column , So creating Manual Schema first
from pyspark.sql.types import *
emp_schema = StructType([
    StructField("id", IntegerType(),True),
    StructField("name",StringType(),True),
    StructField("age",IntegerType(),True),
    StructField("salary",IntegerType(),True),
    StructField("address",StringType(),True),
    StructField("nominee",StringType(),True),
    StructField("corrupted_record",StringType(),True)
])

In [0]:
employee_df = spark.read.format("csv")\
    .option("header", "True")\
    .option("inferschema", "True")\
    .option("mode", "PERMISSIVE")\
    .schema(emp_schema)\
    .load("/Volumes/workspace/default/tables/employee_file.csv")    

employee_df.show(truncate=False)


In [0]:
# save corrupted record in bad_record_path
employee_df = spark.read.format("csv")\
    .option("header", "True")\
    .option("inferschema", "True")\
    .schema(emp_schema)\
    .option("badRecordsPath", "/Volumes/workspace/default/tables/bad_records_path")\
    .load("/Volumes/workspace/default/tables/employee_file.csv")    

employee_df.show(truncate=False)


In [0]:
%fs
ls dbfs:/Volumes/workspace/default/tables/bad_records_path/20251226T103840/bad_records/part-00000-9e4f2699-8c6b-47b5-b3b3-3fa99f9e0344

In [0]:
df = spark.read.format("json").option("header","true").load("dbfs:/Volumes/workspace/default/tables/bad_records_path/20251226T103840/bad_records/part-00000-9e4f2699-8c6b-47b5-b3b3-3fa99f9e0344")
display(df)

In [0]:
dbutils.fs.ls("dbfs:/Volumes/workspace/default/tables/employee_file.csv")

## DAG vs Lagy Evaluation

In [0]:
# serverless compute will not show JOB for security risk because serverless used by many other company also so in spark there a chance to disclose their data so databricks has disabled this feature here
from pyspark.sql.types import *
from pyspark.sql.functions import * 
flight_data = spark.read.format("csv")\
    .option("header", "true")\
    .option("inferSchema", "true")\
    .load("/Volumes/workspace/default/tables/flight_data.csv")   # <--- Read

flight_data_repartition = flight_data.repartition(3)  # <--- WIDE DEPENDENCY (Shuffling)

us_flight_data = flight_data.filter("DEST_COUNTRY_NAME=='United States'") # <--- NARROW DEPENDENCY (No Shuffling)

us_india_data = us_flight_data.filter((col("ORIGIN_COUNTRY_NAME")=='India') |(col("ORIGIN_COUNTRY_NAME")=='Singapore'))        # <--- TRANSFORMATION (Filter)

total_flight_ind_sing = us_india_data.groupby("DEST_COUNTRY_NAME").sum("count") # <-WIDE DEPENDENCY (Grouping/Shuffling)

total_flight_ind_sing.show() # <--- ACTION (Triggers Execution)

## Read JSOn file


In [0]:
#line delimited
df = spark.read.format("JSON").load("/Volumes/workspace/default/tables/line_delimited_json.json").show()

In [0]:
# line-delimited with extra column in 5th row
df = spark.read.format("JSON").load("/Volumes/workspace/default/tables/single_file_json_with_extra_fields.json").show()

In [0]:
# multiline-json with , read 1st data as a dictionary list omited out
df = spark.read.format("JSON").option("multiline","true").load("/Volumes/workspace/default/tables/Multi_line_incorrect.json").show()

In [0]:
# Json with corrupted record, 5th row closed "}" bracket is not there
df = spark.read.format("JSON").load("/Volumes/workspace/default/tables/corrupted_json.json").show(truncate=False)

## Parquet file 

In [0]:
df = spark.read.parquet("/Volumes/workspace/default/tables/part-r-00000-1a9822ba-b8fb-4d8e-844a-ea30d0801b9e.gz.parquet")
display(df)

## write dataframe to disk in spark

In [0]:
# load the file 
from pyspark.sql import SparkSession
df = spark.read.csv("/Volumes/workspace/default/tables/flight_data.csv",header=True,inferSchema=True)
display(df)


In [0]:
# suppose after transformation I want to put this file /Volumes/workspace/default/tables/flight_data.csv in another path . Here no need to define file name automatic file will be generated and will be stored your destination path
df.write.format("CSV")\
    .option("header","true")\
        .option("mode","overwrite")\
            .option("path","/Volumes/workspace/default/tables/write_to_disk")\
                .save()

In [0]:
# want to see the files actually going destination path or not
dbutils.fs.ls("/Volumes/workspace/default/tables/write_to_disk")

In [0]:
# Split the files into 3 parts using repartition
df.repartition(3).write.format("CSV")\
    .option("header","true")\
        .option("mode","overwrite")\
            .option("path","/Volumes/workspace/default/tables/write_to_disk_3_repartition")\
                .save()

## Partitionby & Bucketing

In [0]:
# load the file 
from pyspark.sql import SparkSession
df = spark.read.csv("/Volumes/workspace/default/tables/employee_write_data.csv",header=True,inferSchema=True)
display(df)

In [0]:
# As you see there are multiple data accross different country. So I want country like India/ USA/JAPAN.. data come according to country wise
df.write.format("CSV")\
    .option("header","true")\
    .option("mode","overwrite")\
    .option("path","/Volumes/workspace/default/tables/partition_by_address")\
    .partitionBy("address")\
    .save()

In [0]:
dbutils.fs.ls("/Volumes/workspace/default/tables/partition_by_address")

In [0]:
# Read India data 
df2 = spark.read.csv("dbfs:/Volumes/workspace/default/tables/partition_by_address/address=INDIA/", header="true")
display(df2)

In [0]:
# Simple read
from pyspark.sql import SparkSession
from pyspark.sql.functions import sum
spark = SparkSession.builder.appName("SparkByExamples.com").getOrCreate()
df = spark.read.option("header","true").csv("/Volumes/workspace/default/tables/partition_by_address")
display(df)

In [0]:
# PartitionBy address & gender
df.write.format("CSV")\
    .option("header","true")\
    .option("mode","overwrite")\
    .option("path","/Volumes/workspace/default/tables/partition_by_address_gender")\
    .partitionBy("address","gender")\
    .save()


In [0]:
# create 3 buck , id wise
df.write.format("CSV")\
    .option("header","true")\
    .option("mode","overwrite")\
    .bucketBy(3,"id")\
    .saveAsTable("bucket_by_id_table")

## # how to create dataframe in spark

In [0]:

my_data = [(1,   1),
(2,   1),
(3,   1),
(4,   2),
(5,   1),
(6,   2),
(7,   2)]

# taking both column in string
my_schema = ['id','num']

spark.createDataFrame(data=my_data, schema = my_schema).show()

## dataframe transformations/ Selecting columns in spark

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
# taking both column in string
# String method , selecting column
employee_df.select("name").show()

In [0]:
# Col method 
employee_df.select(col("name")).show()

In [0]:
#  want to see id column increased by +5
employee_df.select(col("id")+5).show()

In [0]:
#Selecting column using expression
employee_df.select(expr("id + 5")).show()

In [0]:
#Selecting column using expression for alias & concat 2 columns
employee_df.select(expr("id as emp_id"),expr("name as emp_name"),expr("concat(name,address)")).show()

In [0]:
#Selecting Multiple Columns
employee_df.select("id","name","age").show()

In [0]:
#Selecting Multiple Columns using col
employee_df.select(col("id"),col("name"),col("age")).show()

In [0]:
#Selecting Multiple Columns using function
employee_df.select("id",col("name"),employee_df['salary'],employee_df.address).show()

In [0]:
#Selecting column using spark SQL
# convert employee_df to temporary view/table 
employee_df.createOrReplaceTempView("employee_tbl")

In [0]:
spark.sql('''
          select * from employee_tbl''').show()

In [0]:
#or
employee_df.select('*').show()

In [0]:
spark.sql('''
          select id,name,age from employee_tbl''').show()

## dataframe transformations/ Selecting columns in spark PART-2

In [0]:
# 1. Aliasing 
employee_df.select(col("id").alias("emp_id"),"name","age").show()

In [0]:
# 2. filter / where 
# select >150,000
employee_df.filter(col("salary")>150000).show()

In [0]:
# using where
employee_df.where(col("salary")>150000).show()

In [0]:
#filter using multiple condition
employee_df.filter((col("salary")>150000) & (col("age")>18)).show()

In [0]:
# 3. Literal
employee_df.select("*",lit("kumar")).show()
employee_df.select("*",lit("kumar").alias("surname")).show()

In [0]:
# 4. Adding columns
# withColumn create new column/ if existing there it can take and override
employee_df.withColumn("sur_name",lit("singh")).show()
# employee_df.withColumn("bonus",col("salary")*0.1).show()

In [0]:
# 5. Renaming columns
employee_df.withColumnRenamed("id","emp_id").show()

In [0]:
# 6. casting data types
employee_df.printSchema()

#converting id string and salary long
employee_df.withColumn("id",col("id").cast("string"))\
           .withColumn("salary",col("salary").cast("long"))\
           .printSchema()

In [0]:
# 7. Removing columns
employee_df.drop("id").show()

#or
# dropping id usinf string method & name using column method
employee_df.drop("id",col("name")).show()

In [0]:
#Using Spark SQL
# 1. Aliasing 
# 2. filter / where: where salary>150000 and age<18
# 3. Literal: "kumar" as lastname
# 4. Adding columns: concat(name,lastname) as fullname
# 5. Renaming columns: id as emp_id
# 6. casting data types: cast(id as string)
# 7. Removing columns: dont include particular column in select statement 
spark.sql('''
          select *, "kumar" as lastname, concat(name,lastname) as fullname, id as emp_id, cast(id as string) from employee_tbl where salary>150000 and age<18''').printSchema()

In [0]:
#Add 2 columns #Mphasis
from pyspark.sql.functions import *
from pyspark.sql.types import *

data = [("Sohom", "Pradhan"),
        ("Namita","sen")]
schema = ["firstname","lastname"]
df = spark.createDataFrame(data=data, schema=schema)
display(df)
df2 = df.withColumn("fullname",concat("firstname","lastname"))
display(df2)
        

## Union & UnionAll

In [0]:
data=[(10 ,'Anil',50000, 18),
(11 ,'Vikas',75000,  16),
(12 ,'Nisha',40000,  18),
(13 ,'Nidhi',60000,  17),
(14 ,'Priya',80000,  18),
(15 ,'Mohit',45000,  18),
(16 ,'Rajesh',90000, 10),
(17 ,'Raman',55000, 16),
(18 ,'Sam',65000,   17),
(17 ,'Raman',55000, 16)]
schema = ["id","name","salary","mngr_id"]
manager_df = spark.createDataFrame(data=data,schema=schema)
manager_df.show()
manager_df.count()

In [0]:
data1=[(19 ,'Sohan',50000, 18),
(20 ,'Sima',75000,  17)]

schema1 = ["id","name","salary","mngr_id"]
manager_df1 = spark.createDataFrame(data=data1,schema=schema1)
manager_df1.show()
manager_df1.count()

In [0]:
# pyspark union/unionAll same and it will not take distinct only take in sql 
manager_df.union(manager_df1).show()
manager_df.union(manager_df1).count()

manager_df.unionAll(manager_df1).show()
manager_df.unionAll(manager_df1).count()

In [0]:
# union/UnionAll using Spark SQL
manager_df1.createOrReplaceTempView("manager_df1_tbl")
manager_df.createOrReplaceTempView("manager_df_tbl")


In [0]:
# duplicate row removed
spark.sql('''
          select * from manager_df_tbl
          union
          select * from manager_df1_tbl ''').show()
spark.sql('''
          select * from manager_df_tbl
          union
          select * from manager_df1_tbl ''').count()          

In [0]:
# duplicate value not removed in union all
spark.sql('''
          select * from manager_df_tbl
          union all
          select * from manager_df1_tbl ''').show()
spark.sql('''
          select * from manager_df_tbl
          union all
          select * from manager_df1_tbl ''').count()          

In [0]:
#Error: for this case dataframe will not be created it will throw error as data is not same
wrong_column_data=[(19 ,50000, 18,'Sohan'),
(20 ,75000,  17,'Sima')]
wrong_schema = ['id',"salary","mngr_id",'name','bonus']
wrong_manager_df = spark.createDataFrame(data=wrong_column_data,schema=wrong_schema)
wrong_manager_df.show()

In [0]:
wrong_column_data=[(19 ,50000, 18,'Sohan',10),
(20 ,75000,  17,'Sima',20)]
wrong_schema = ['id',"salary","mngr_id",'name','bonus']
wrong_manager_df = spark.createDataFrame(data=wrong_column_data,schema=wrong_schema)
wrong_manager_df.show()

In [0]:
# for different no of columns union will not be performed so have to take only selected column
wrong_manager_df.union(manager_df).show()

In [0]:
wrong_manager_df.select("id","mngr_id","name").union(manager_df).show()

In [0]:
#repartition vs coalesce
# display(flight_df) 

# see how much partition is there
flight_df.rdd.getNumPartitions()

In [0]:
from pyspark.sql.functions import spark_partition_id
from pyspark.sql.types import *
partition_flight_df = flight_df.repartition(4)


In [0]:
from pyspark.sql.functions import spark_partition_id
# spark_partiton_id available in pyspark type or function
partition_flight_df = partition_flight_df.withColumn(
    "partitionID",
    spark_partition_id()
)

result_df = partition_flight_df.groupby("partitionID").count()
display(result_df)

In [0]:
# repartition you can do with single column also
partition_on_column = flight_df.repartition(300,"ORIGIN_COUNTRY_NAME")

# in table total 250 records are there so how it can be done with 300 partition so in the some places it will put null. But it is not a good approach as multipple partiton is happening



In [0]:
# coalesce
# part-1
# before doing coalesce lets make the df with 8 partition then will do the coalse with 3 partition
coalesce_flight_df = flight_df.repartition(8)

In [0]:
# part-2
from pyspark.sql.functions import spark_partition_id
coalesce_flight_df = coalesce_flight_df.withColumn("partitionId",spark_partition_id())
result = coalesce_flight_df.groupby("partitionId").count()
display(result)


In [0]:
#part-3 final part coalesce
three_coalesce_df = result.coalesce(3)



In [0]:
from pyspark.sql.functions import spark_partition_id
three_coalesce_df = three_coalesce_df.withColumn("partitionId",spark_partition_id())
result4 = three_coalesce_df.groupby("partitionId").count()
display(result4)

##  if else in pyspark


In [0]:
emp_data = [
(1,'manish',26,20000,'india','IT'),
(2,'rahul',None,40000,'germany','engineering'),
(3,'pawan',12,60000,'india','sales'),
(4,'roshini',44,None,'uk','engineering'),
(5,'raushan',35,70000,'india','sales'),
(6,None,29,200000,'uk','IT'),
(7,'adam',37,65000,'us','IT'),
(8,'chris',16,40000,'us','sales'),
(None,None,None,None,None,None),
(7,'adam',37,65000,'us','IT')]
schema = ["id","name","age","salary","country","dept"]
emp_df = spark.createDataFrame(data=emp_data,schema=schema)
emp_df.show()

In [0]:
# Scenerio-1: using condition if-else : create column adult age <18 -> "No", age>18 ->"yes", otherwise no value

emp_df.withColumn("adult",when(col("age")<18,"no")
      .when(col("age")>18,"yes")
      .otherwise("Novalue")).show()

In [0]:
#Scenerio-2 , "When". if any age column is null make it 19 otherwise age value as well. And >18:yes, <18:NO
emp_df.withColumn("age",when(col("age").isNull(),lit(19)).otherwise(col("age")))\
    .withColumn("adult_or_not",when(col("age")>18,"yes")
    .otherwise("No")).show()

In [0]:
# 0-<18: minor, 18-30: Mid, Major
emp_df.withColumn("age_wise",when((col("age")>0) & (col("age")<18),"Minor")
                             .when((col("age")>18) & (col("age")<30),"Mid")
                             .otherwise("Major")).show()

In [0]:
#if/when condition using spark sql
# create a column name "adult": end as adult

emp_df.createOrReplaceTempView("emp_table")
spark.sql('''select *,
          case when age<18 then 'Minor'
          when age>18 then 'Major'
          else 'novalue'
          end as adult
          from emp_table''').show()

## 
Unique & sorted records

In [0]:
data=[(10 ,'Anil',50000, 18),
(11 ,'Vikas',75000,  16),
(12 ,'Nisha',40000,  18),
(13 ,'Nidhi',60000,  17),
(14 ,'Priya',80000,  18),
(15 ,'Mohit',45000,  18),
(16 ,'Rajesh',90000, 10),
(17 ,'Raman',55000, 16),
(18 ,'Sam',65000,   17),
(15 ,'Mohit',45000,  18),
(13 ,'Nidhi',60000,  17),      
(14 ,'Priya',90000,  18),  
(18 ,'Sam',65000,   17)]

schema = ["id","name","sal","mngr_id"]
mngr_df = spark.createDataFrame(data=data,schema=schema)
mngr_df.show()
mngr_df.count()

In [0]:
# Display duplicate Records
display(mngr_df.groupBy(mngr_df.columns).count().filter(col("count")>1))


In [0]:
# Remove duplicate records
mngr_df.dropDuplicates().show()
        # OR
mngr_df.distinct().show()

In [0]:
# Remove duplicate records based on specific columns.
# deopDuplicate vs distinct: dropDuplicates remove duplicate records based on condition and give the full record but distinct give particular column ex id/name in output  

# mngr_df.dropDuplicates(['id','name']).show()
            #or
mngr_df.select("id","name").distinct().show()


In [0]:
# why we need to save the dataframe value in new dataframe for save and operate further because 1st dataframe created is immutable whether it can be full update, 

In [0]:
#Sort
mngr_df.sort(col("sal")).show()
#Sort on Single column Sal, desc 
mngr_df.sort(col("sal").desc()).show()
mngr_df.sort(col("sal").desc(),col("id").desc()).show()
#Sort on double column sal & name
mngr_df.sort(col("sal").desc(),col("id").asc()).show()

## aggregate function

In [0]:
emp_data = [
(1,'manish',26,20000,'india','IT'),
(2,'rahul',None,40000,'germany','engineering'),
(3,'pawan',12,60000,'india','sales'),
(4,'roshini',44,None,'uk','engineering'),
(5,'raushan',35,70000,'india','sales'),
(6,None,29,200000,'uk','IT'),
(7,'adam',37,65000,'us','IT'),
(8,'chris',16,40000,'us','sales'),
(None,None,None,None,None,None),
(7,'adam',37,65000,'us','IT')]
schema = ["id","name","age","salary","country","dept"]
emp_df = spark.createDataFrame(data=emp_data,schema=schema)
emp_df.show()

In [0]:
# count all columns, do not remove null
emp_df.count()

In [0]:
# count on single columns, remove null
# emp_df.select(col("name")).count()
# 10
emp_df.select(count("name")).show()
# 8

In [0]:
emp_df.select(sum("salary"),max("salary"),min("salary")).show()
emp_df.select(sum("salary").alias("total_salary"),max("salary").alias("maxm_salary"),min("salary")).show()

In [0]:
# convert to integer
avg("salary").cast("int")

## Group By in Spark

In [0]:
data = [(1,'manish',50000,'IT'),
(2,'vikash',60000,'sales'),
(3,'raushan',70000,'marketing'),
(4,'mukesh',80000,'IT'),
(5,'pritam',90000,'sales'),
(6,'nikita',45000,'marketing'),
(7,'ragini',55000,'marketing'),
(8,'rakesh',100000,'IT'),
(9,'aditya',65000,'IT'),
(10,'rahul',50000,'marketing')]
schema = ["id","name","sal","dept"]
emp_df = spark.createDataFrame(data=data,schema=schema)
# emp_df.show()

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
from pyspark.sql.functions import *
#What is the total salary given to it's employee department wise?
# groupBy: on which column, agg: aggregate 
emp_df.groupBy("dept").agg(sum("sal")).show()
# df = emp_df.groupBy("dept").agg(max("sal").alias("heighest_sal"))
# if someone wants it name wise on that time window function have to use  
# Added Filter with Aggregation  
# result_df = emp_df.groupBy("dept").agg(sum('sal').alias("sum_of_sal")).filter(col("sum_of_sal")>150000)

In [0]:
# group by using spark sql
emp_df.createOrReplaceTempView("emp_table")
spark.sql('''
          select dept, sum(sal) from emp_table
          group by dept ''').show()

In [0]:
from pyspark.sql.functions import explode
data = [
    ("James", ["Java", "Python", "SQL"]),
    ("Jill", ["Java", "Python"])
]

df = spark.createDataFrame(data, ["User", "Skills"])
display(df)

df.withColumn("Skill", explode("Skills")).show()